# NPF subtype connectome analysis

Connectivity analysis of NPF neuron subtypes (DM, L1, P1, P2) in the FlyWire/Codex adult *Drosophila*
brain connectome (snapshot 783).

**Naming convention** (NPF subtypes = order 0):

| Label | Meaning |
|---|---|
| `in1` | direct presynaptic partners |
| `in2` | presynaptic partners of `in1` |
| `out1` | direct postsynaptic partners |
| `out2` | postsynaptic partners of `out1` |

Level codes used by the plotting methods: `0`=NPF, `+1`=out1, `+2`=out2, `-1`=in1, `-2`=in2.

## 0. Imports and plot styling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from venn import venn
import seaborn as sns

In [ ]:
sns.set_theme(style='white')

andy_theme = {'axes.grid': True, 'grid.linestyle': '--',
              'legend.framealpha': 1, 'legend.facecolor': 'white', 'legend.shadow': False,
              'legend.fontsize': 14, 'legend.title_fontsize': 14,
              'xtick.labelsize': 8, 'ytick.labelsize': 8,
              'axes.labelsize': 12, 'axes.titlesize': 16, 'figure.dpi': 100}
plt.rcParams.update(andy_theme)

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

## 1. The core classes

Three classes implement the analysis:

1. `FafbData` — loads and holds the four FlyWire data tables.
2. `Neuralgroup` — one set of neurons (e.g. DM) + its cached up/down-stream connections.
3. `Connectsets` — relationships *between* several Neuralgroups, expanded to 2nd order.

### 1a. `FafbData` — load the four FlyWire tables once

Reads each table once; every later step reuses `fafb.connections`, `fafb.neurons`, etc.

In [ ]:
class FafbData:
    """Holds the four FlyWire/Codex tables (snapshot 783)."""

    def __init__(self, connections_csv, neuropil_csv, classification_csv, neurons_csv):
        # connections: pre_root_id, post_root_id, neuropil, syn_count, nt_type
        self.connections = pd.read_csv(connections_csv)

        self.neuropils = (pd.read_csv(neuropil_csv)[['root_id', 'input synapses', 'output synapses']]
                          .rename(columns={'input synapses': 'input_synapses',
                                           'output synapses': 'output_synapses'}))

        cls = pd.read_csv(classification_csv)
        self.side = cls[['root_id', 'side']]
        self.classes = cls[['root_id', 'super_class', 'class']]

        self.neurons = pd.read_csv(neurons_csv)[['root_id', 'nt_type', 'group']]

        self.neurons_data = (self.neurons
                             .merge(self.side, on='root_id', how='outer')
                             .merge(self.neuropils, on='root_id', how='outer'))

### 1b. `Neuralgroup` — one set of neurons + its connectivity

Given a list of neuron IDs (e.g. the DM neurons), stores:

- `downconnections` / `upconnections` — every synaptic edge *out of* / *into* the group.
- `postneurons` / `preneurons` — the unique partner neurons downstream / upstream.
- `region` — the set of neuropils this group lives in (used to tag connections local vs. nonlocal).

In [ ]:
class Neuralgroup:
    """A set of neurons plus their cached up/down-stream connectivity."""

    def __init__(self, subtype_df, fafbdata, name, min_syn=5):
        self.name = name
        self.min_syn_count = min_syn
        self.fafbdata = fafbdata

        df = subtype_df.loc[:, ~subtype_df.columns.str.contains('^Unnamed')].copy()
        self.neurons = df[['root_id']].drop_duplicates().reset_index(drop=True)

        self.region = self._detect_region(fafbdata.neurons)

        self.downconnections = self._edges(fafbdata.connections, 'down', min_syn)
        self.upconnections   = self._edges(fafbdata.connections, 'up',   min_syn)

        self.postneurons = self._partners(self.downconnections, 'post_root_id')
        self.preneurons  = self._partners(self.upconnections,  'pre_root_id')

        # exclude within-group synapses so downflow/upflow are external partners only
        self_ids = set(self.neurons['root_id'])
        self.downflow = self.postneurons[~self.postneurons['root_id'].isin(self_ids)].reset_index(drop=True)
        self.upflow   = self.preneurons[~self.preneurons['root_id'].isin(self_ids)].reset_index(drop=True)

    def _detect_region(self, neurons_df):
        """Return the set of neuropils this group's neurons live in (from the `group` string)."""
        groups = neurons_df.loc[neurons_df['root_id'].isin(self.neurons['root_id']), 'group'].dropna()
        regions = set()
        for g in groups:
            regions.update(str(g).split('.'))
        return regions

    def _projection(self, neuropil_side):
        """Label a connection 'local' (inside the group's own neuropils) or 'nonlocal'."""
        if pd.isna(neuropil_side):
            return 'nonlocal'
        npil = neuropil_side.replace('_R', '').replace('_L', '')
        return 'local' if npil in self.region else 'nonlocal'

    def _edges(self, connections_df, direction, min_syn):
        """Every connection out of (down) or into (up) this group, above the synapse threshold."""
        key = 'pre_root_id' if direction == 'down' else 'post_root_id'
        conn = connections_df[['pre_root_id', 'post_root_id', 'neuropil', 'syn_count', 'nt_type']]
        edges = (self.neurons
                 .merge(conn, left_on='root_id', right_on=key, how='inner')
                 .drop(columns='root_id')
                 .query('syn_count >= @min_syn')
                 .copy())
        edges['locality'] = edges['neuropil'].apply(self._projection)
        return edges.reset_index(drop=True)

    @staticmethod
    def _partners(edges, col):
        """Collapse an edge list to the unique partner ids in column `col`."""
        if edges.empty:
            return pd.DataFrame({'root_id': pd.Series(dtype='int64')})
        return (edges.groupby(col)
                .agg({'syn_count': 'sum'})
                .reset_index()[[col]]
                .rename(columns={col: 'root_id'}))

    def DownstreamNeurons(self, connections_df=None, min_syn=None):
        # reuse the cached edges when the requested threshold isn't stricter than the cache
        if min_syn is None:
            min_syn = self.min_syn_count
        if connections_df is None and min_syn >= self.min_syn_count:
            return self.downconnections.query('syn_count >= @min_syn').reset_index(drop=True)
        src = connections_df if connections_df is not None else self.fafbdata.connections
        return self._edges(src, 'down', min_syn)

    def UpstreamNeurons(self, connections_df=None, min_syn=None):
        if min_syn is None:
            min_syn = self.min_syn_count
        if connections_df is None and min_syn >= self.min_syn_count:
            return self.upconnections.query('syn_count >= @min_syn').reset_index(drop=True)
        src = connections_df if connections_df is not None else self.fafbdata.connections
        return self._edges(src, 'up', min_syn)

### 1c. `Connectsets` — connections *between* groups, expanded to 2nd order

Given the four subtype Neuralgroups, finds:

- **order 1** partners (`out1_groups` / `in1_groups`) — direct partners of each subtype.
- **order 2** partners (`out2_groups` / `in2_groups`) — partners of those partners.

NPF-subtype neurons and neurons already counted at a lower order are excluded. Also provides
the heatmap and Venn plotting methods.

In [ ]:
class Connectsets:
    """Multi-order connectivity between a list of Neuralgroups."""

    def __init__(self, neuralgroup_list, fafbdata,
                 downstream_order=1, upstream_order=0,
                 order1_min_syn=5, order2_min_syn=10):
        self.maingroups = {ng.name: ng for ng in neuralgroup_list}
        self.fafbdata = fafbdata
        self.downstream_order = downstream_order
        self.upstream_order = upstream_order
        self.order1_min_syn = order1_min_syn
        self.order2_min_syn = order2_min_syn

        self._all_main_ids = set(pd.concat([ng.neurons for ng in neuralgroup_list])['root_id'])

        self.out1_connections, self.out1_groups = {}, {}
        self.out2_connections, self.out2_groups = {}, {}
        self.in1_connections,  self.in1_groups  = {}, {}
        self.in2_connections,  self.in2_groups  = {}, {}

        if downstream_order >= 1:
            self.out1_connections, self.out1_groups = self._first_order('down', order1_min_syn)
        if downstream_order >= 2:
            self.out2_connections, self.out2_groups = self._second_order('down', order2_min_syn)
        if upstream_order >= 1:
            self.in1_connections, self.in1_groups = self._first_order('up', order1_min_syn)
        if upstream_order >= 2:
            self.in2_connections, self.in2_groups = self._second_order('up', order2_min_syn)

    def _first_order(self, stream, min_syn):
        """Direct partners of each main subtype (excluding other NPF subtypes)."""
        connectivity, groups = {}, {}
        for name, ng in self.maingroups.items():
            if stream == 'down':
                edges, partner_col = ng.DownstreamNeurons(min_syn=min_syn), 'post_root_id'
            else:
                edges, partner_col = ng.UpstreamNeurons(min_syn=min_syn), 'pre_root_id'
            edges = edges[~edges[partner_col].isin(self._all_main_ids)].reset_index(drop=True)
            partners = Neuralgroup._partners(edges, partner_col)
            partners = partners[~partners['root_id'].isin(self._all_main_ids)]
            groups[name] = Neuralgroup(partners, self.fafbdata, name, min_syn=min_syn)
            connectivity[name] = edges
        return connectivity, groups

    def _second_order(self, stream, min_syn):
        """Partners of the order-1 partners, excluding NPF subtypes and order-1 neurons."""
        order1_groups = self.out1_groups if stream == 'down' else self.in1_groups
        if not order1_groups:
            raise ValueError('First-order groups not built; set the order >= 1.')
        all_order1 = set(pd.concat([ng.neurons for ng in order1_groups.values()])['root_id'])
        connectivity, groups = {}, {}
        for name, ng in order1_groups.items():
            if stream == 'down':
                edges, partner_col = ng.DownstreamNeurons(min_syn=min_syn), 'post_root_id'
            else:
                edges, partner_col = ng.UpstreamNeurons(min_syn=min_syn), 'pre_root_id'
            edges = edges[~edges[partner_col].isin(self._all_main_ids)]
            edges = edges[~edges[partner_col].isin(all_order1)]
            edges = edges.drop_duplicates().reset_index(drop=True)
            partners = Neuralgroup._partners(edges, partner_col)
            groups[name] = Neuralgroup(partners, self.fafbdata, name, min_syn=min_syn)
            connectivity[name] = edges
        return connectivity, groups

    def _groups_at(self, lvl):
        """Translate a level code (0, +/-1, +/-2) into the matching dict of groups."""
        mapping = {0: self.maingroups, 1: self.out1_groups, 2: self.out2_groups,
                   -1: self.in1_groups, -2: self.in2_groups}
        groups = mapping.get(lvl)
        if not groups:
            raise ValueError(f'No groups available at level {lvl}; check the order settings.')
        return groups

    def Intercon_heatmap(self, pre_lvl=0, post_lvl=0, min_syn=5,
                         xlabel='Downstream Cells', ylabel='Upstream Cells', title='',
                         numfontsize=18, lblfontsize=20, tickfontsize=18, barlblsize=16,
                         cmap='viridis', ax=None):
        """Heatmap of total synapses from each `pre_lvl` group to each `post_lvl` group."""
        pre_groups, post_groups = self._groups_at(pre_lvl), self._groups_at(post_lvl)
        mat = np.zeros((len(pre_groups), len(post_groups)), dtype=int)
        for i, pre in enumerate(pre_groups.values()):
            edges = pre.downconnections.query('syn_count >= @min_syn')
            for j, post in enumerate(post_groups.values()):
                ids = set(post.neurons['root_id'])
                mat[i, j] = int(edges.loc[edges['post_root_id'].isin(ids), 'syn_count'].sum())
        row_labels, col_labels = list(pre_groups.keys()), list(post_groups.keys())
        if ax is None:
            ax = plt.gca()
        sns.heatmap(mat, cmap=cmap, annot=True, fmt='d',
                    xticklabels=col_labels, yticklabels=row_labels,
                    square=True, vmin=0, annot_kws={'fontsize': numfontsize}, ax=ax)
        ax.set_title(title)
        ax.set_ylabel(ylabel, fontsize=lblfontsize)
        ax.set_xlabel(xlabel, fontsize=lblfontsize)
        ax.tick_params(labelsize=tickfontsize)
        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=barlblsize)
        plt.show()
        return mat, row_labels, col_labels

    def Venn(self, lvl=0, neuropil='input', min_syn=5, legend=True,
             colors=('#cf4848', 'orange', '#3489eb', 'purple'),
             figsize=(8, 8), fontsize=18):
        """Venn diagram of how much the groups at `lvl` share partners."""
        groups = self._groups_at(lvl)
        if neuropil == 'input':
            crossover = {n: set(ng.UpstreamNeurons(min_syn=min_syn)['pre_root_id'].unique())
                         for n, ng in groups.items()}
        else:
            crossover = {n: set(ng.DownstreamNeurons(min_syn=min_syn)['post_root_id'].unique())
                         for n, ng in groups.items()}
        ax = venn(crossover, cmap=ListedColormap(list(colors)), figsize=figsize, fontsize=fontsize)
        leg = ax.get_legend()
        if leg and not legend:
            leg.remove()
        plt.show()
        return crossover

## 2. Load the data

Requires the four FlyWire snapshot-783 tables and the four subtype id lists in the same
folder as this notebook.

In [ ]:
fafb = FafbData(connections_csv='connections_783.csv.gz',
                neuropil_csv='neuropil_synapse_table_783.csv.gz',
                classification_csv='classification_783.csv.gz',
                neurons_csv='neurons_783.csv.gz')

DM = Neuralgroup(pd.read_csv('DM_r_id.csv'), fafb, 'DM', min_syn=5)
L1 = Neuralgroup(pd.read_csv('L1_r_id.csv'), fafb, 'L1', min_syn=5)
P1 = Neuralgroup(pd.read_csv('P1_r_id.csv'), fafb, 'P1', min_syn=5)
P2 = Neuralgroup(pd.read_csv('P2_r_id.csv'), fafb, 'P2', min_syn=5)
subtypes = [DM, L1, P1, P2]

In [ ]:
print('DM has', len(DM.neurons), 'neurons')
print('DM makes', len(DM.downconnections), 'output edges (>=5 syn)')
DM.downconnections.head()

## 3. Downstream (postsynaptic) analysis

`downstream_order=2` builds both `out1` (direct targets) and `out2`.

In [ ]:
down = Connectsets(subtypes, fafb, downstream_order=2, upstream_order=0,
                   order1_min_syn=5, order2_min_syn=10)

for name in ['DM', 'L1', 'P1', 'P2']:
    down.out1_connections[name].to_csv(f'{name}_out1.csv', index=False)
    down.out2_connections[name].to_csv(f'{name}_out2.csv', index=False)

In [ ]:
# How much do the four subtypes share downstream targets?
down.Venn(lvl=0, neuropil='output', min_syn=5, fontsize=24, legend=False)

In [ ]:
# How much do the four subtypes share 2nd order downstream targets?
down.Venn(lvl=1, neuropil='output', min_syn=10, fontsize=24, legend=False)

In [ ]:
# Synapses between the subtypes themselves (do they talk to each other?)
down.Intercon_heatmap(pre_lvl=0, post_lvl=0,
                      xlabel='Downstream NPF', ylabel='Upstream NPF',
                      title='NPF subtype -> subtype synapses')

In [ ]:
# out1 -> out1: do the direct targets of different subtypes interconnect?
down.Intercon_heatmap(pre_lvl=1, post_lvl=1, min_syn=10,
                      xlabel='Downstream out1', ylabel='Upstream out1', title='out1 -> out1')

In [ ]:
# subtype -> out1: do subtypes and the direct targets of different subtypes interconnect?
down.Intercon_heatmap(pre_lvl=0, post_lvl=1, min_syn=5,
                      xlabel='Downstream out1', ylabel='Upstream subtypes', title='subtype -> out1')

In [ ]:
# out2 -> out2: do the 2nd order targets of different subtypes interconnect?
down.Intercon_heatmap(pre_lvl=2, post_lvl=2, min_syn=10,
                      xlabel='Downstream out2', ylabel='Upstream out2', title='out2 -> out2')

## 4. Upstream (presynaptic) analysis

Same as section 3, but expanding backwards to what feeds into the subtypes.

In [ ]:
up = Connectsets(subtypes, fafb, downstream_order=0, upstream_order=2,
                 order1_min_syn=5, order2_min_syn=10)

for name in ['DM', 'L1', 'P1', 'P2']:
    up.in1_connections[name].to_csv(f'{name}_in1.csv', index=False)
    up.in2_connections[name].to_csv(f'{name}_in2.csv', index=False)

In [ ]:
up.Venn(lvl=0, neuropil='input', min_syn=5, fontsize=24, legend=False)

In [ ]:
up.Venn(lvl=-1, neuropil='input', min_syn=10, fontsize=24, legend=False)

In [ ]:
# in1 -> in1: do the direct sources of different subtypes interconnect?
up.Intercon_heatmap(pre_lvl=-1, post_lvl=-1, min_syn=10,
                      xlabel='Downstream in1', ylabel='Upstream in1', title='in1 -> in1')

In [ ]:
# in1 -> subtype: do the direct sources of different subtypes and the subtypes interconnect?
up.Intercon_heatmap(pre_lvl=-1, post_lvl=0, min_syn=5,
                      xlabel='Downstream subtypes', ylabel='Upstream in1', title='in1 -> subtype')

In [ ]:
# in2 -> in2: do the 2nd order sources of different subtypes interconnect?
up.Intercon_heatmap(pre_lvl=-2, post_lvl=-2, min_syn=10,
                      xlabel='Downstream in2', ylabel='Upstream in2', title='in2 -> in2')

## 5. Extra analyses to distinguish subtype function

Each function below builds a summary table, plots it, and returns the table for inspection/export.

### 5a. Neurotransmitter composition

What neurotransmitter each subtype releases, as a fraction of its output synapses.

In [ ]:
def nt_composition(groups, fafb, by='output'):
    rows = []
    nt_lookup = fafb.neurons.set_index('root_id')['nt_type']
    for name, ng in groups.items():
        if by == 'output':
            s = ng.downconnections.groupby('nt_type')['syn_count'].sum()
        elif by == 'input':
            s = ng.upconnections.groupby('nt_type')['syn_count'].sum()
        elif by == 'partner_in':
            partners = ng.upflow['root_id'].map(nt_lookup)
            s = partners.value_counts()
        frac = (s / s.sum()).rename(name)
        rows.append(frac)
    comp = pd.concat(rows, axis=1).fillna(0).T
    comp.plot(kind='bar', stacked=True, figsize=(7, 4), colormap='Set2')
    plt.ylabel('fraction of synapses'); plt.title(f'NT composition ({by})')
    plt.legend(title='nt_type', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout(); plt.show()
    return comp

groups0 = {ng.name: ng for ng in subtypes}
nt_out = nt_composition(groups0, fafb, by='output')
nt_out

### 5b. Neuropil innervation profile (log-scaled heatmap)

Subtype x neuropil table of where synapses land — which brain regions each subtype targets.
The color scale is logarithmic so neuropils with few synapses stay visible next to dominant
ones (zeros are left blank). Use `log=False` for a linear scale, or `normalize=True` (linear
only) to compare profile shapes as fractions.

In [ ]:
def neuropil_matrix(groups, direction='output', min_syn=5):
    """subtype x neuropil table of total synapses (raw counts). direction: 'output' or 'input'."""
    cols = {}
    for name, ng in groups.items():
        edges = ng.downconnections if direction == 'output' else ng.upconnections
        cols[name] = edges.query('syn_count >= @min_syn').groupby('neuropil')['syn_count'].sum()
    return pd.DataFrame(cols).fillna(0)


def neuropil_profile(groups, direction='output', top=25, log=True, normalize=False, min_syn=5):
    from matplotlib.colors import LogNorm
    mat = neuropil_matrix(groups, direction, min_syn)
    mat = mat.loc[mat.sum(axis=1).sort_values(ascending=False).index[:top]]
    plt.figure(figsize=(5, max(4, 0.32 * len(mat))))
    if log:
        data = mat.replace(0, np.nan)
        vmin = max(1, np.nanmin(data.values))
        sns.heatmap(data, cmap='magma', norm=LogNorm(vmin=vmin, vmax=np.nanmax(data.values)),
                    linewidths=.3, linecolor='lightgray')
        scale = 'log count'
    else:
        if normalize:
            mat = mat / mat.sum(axis=0)
        sns.heatmap(mat, cmap='magma')
        scale = 'fraction' if normalize else 'count'
    plt.gca().grid(False)
    plt.title(f'Neuropil {direction} synapses ({scale})')
    plt.xlabel('NPF subtype', fontsize=16); plt.ylabel('Neuropil', fontsize=16); plt.xticks(fontsize=14); plt.yticks(fontsize=12); plt.tight_layout(); plt.show()
    return mat

neuropil_out = neuropil_profile(groups0, direction='output', top=25, log=True)
neuropil_out

### 5b-2. Top output neuropils per subtype (ranked bar charts)

One horizontal bar chart per subtype showing its strongest output neuropils (log x-axis).

In [ ]:
def neuropil_top_bars(groups, direction='output', top=10, min_syn=5, log=True):
    n = len(groups)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 5))
    for ax, (name, ng) in zip(np.atleast_1d(axes), groups.items()):
        edges = ng.downconnections if direction == 'output' else ng.upconnections
        s = (edges.query('syn_count >= @min_syn').groupby('neuropil')['syn_count'].sum()
             .sort_values().tail(top))
        ax.barh(s.index, s.values, color='steelblue')
        if log:
            ax.set_xscale('log')
        ax.set_title(name); ax.set_xlabel(f'{direction} synapses')
    plt.tight_layout(); plt.show()

neuropil_top_bars(groups0, direction='output', top=10)

### 5b-3. Grouped bar: subtypes compared across the top neuropils

The same neuropils side by side for all four subtypes.

In [ ]:
def neuropil_grouped_bar(groups, direction='output', top=12, min_syn=5, log=True):
    mat = neuropil_matrix(groups, direction, min_syn)
    mat = mat.loc[mat.sum(axis=1).sort_values(ascending=False).index[:top]]
    ax = mat.plot(kind='bar', figsize=(12, 5), width=0.82, colormap='Set2', logy=log)
    ax.set_ylabel(f'{direction} synapses' + (' (log)' if log else ''))
    ax.set_xlabel('neuropil'); ax.set_title(f'{direction} synapses by neuropil (top {top})')
    ax.legend(title='subtype'); plt.tight_layout(); plt.show()
    return mat

neuropil_grouped_bar(groups0, direction='output', top=12)

### 5b-4. Local vs nonlocal output

Fraction of each subtype's output synapses inside its own neuropils (`local`) vs outside
(`nonlocal`), using the `locality` tag computed for every edge.

In [ ]:
def locality_bar(groups, direction='output', min_syn=5):
    cols = {}
    for name, ng in groups.items():
        edges = (ng.downconnections if direction == 'output' else ng.upconnections).query('syn_count >= @min_syn')
        cols[name] = edges.groupby('locality')['syn_count'].sum()
    df = pd.DataFrame(cols).fillna(0).T
    frac = df.div(df.sum(axis=1), axis=0)
    frac.plot(kind='bar', stacked=True, figsize=(7, 4), colormap='coolwarm')
    plt.ylabel(f'fraction of {direction} synapses'); plt.title(f'Local vs nonlocal {direction}')
    plt.legend(title='locality', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout(); plt.show()
    return df

locality_bar(groups0, direction='output')

### 5b-5. Output spread / diversity

How many neuropils each subtype reaches, and how evenly output is spread. Shannon entropy
(bits) is converted to an effective number of neuropils (2^entropy): output concentrated in
one region scores ~1, output spread evenly over many regions scores high.

In [ ]:
def neuropil_diversity(groups, direction='output', min_syn=5):
    recs = []
    for name, ng in groups.items():
        edges = ng.downconnections if direction == 'output' else ng.upconnections
        s = edges.query('syn_count >= @min_syn').groupby('neuropil')['syn_count'].sum()
        p = s / s.sum()
        H = -(p * np.log2(p)).sum()
        recs.append({'subtype': name, 'n_neuropils': len(s),
                     'entropy_bits': H, 'effective_neuropils': 2 ** H})
    df = pd.DataFrame(recs).set_index('subtype')
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    df['n_neuropils'].plot(kind='bar', ax=axes[0], color='teal')
    axes[0].set_title('# neuropils reached'); axes[0].set_ylabel('count')
    df['effective_neuropils'].plot(kind='bar', ax=axes[1], color='darkorange')
    axes[1].set_title('effective # neuropils (2^entropy)'); axes[1].set_ylabel('effective count')
    plt.tight_layout(); plt.show()
    return df

neuropil_diversity(groups0, direction='output')

### 5b-6. Input neuropil heatmap (where each subtype *receives* input)

Same as 5b, for incoming synapses (`direction='input'`).

In [ ]:
neuropil_in = neuropil_profile(groups0, direction='input', top=25, log=True)
neuropil_in

### 5b-7. Neuropil heatmap for second-order output (out2)

Neuropils that the out1 -> out2 step runs through for each subtype, using
`down.out2_connections` (needs `down` built with `downstream_order=2`, section 3). For the
upstream version pass `up.in2_connections`.

In [ ]:
def neuropil_matrix_conn(conn_dict, min_syn=5):
    """subtype x neuropil synapse table from a dict of edge DataFrames
    (e.g. down.out2_connections / up.in2_connections). Each must have 'neuropil' + 'syn_count'."""
    cols = {}
    for name, edges in conn_dict.items():
        cols[name] = edges.query('syn_count >= @min_syn').groupby('neuropil')['syn_count'].sum()
    return pd.DataFrame(cols).fillna(0)


def neuropil_heatmap(mat, title='Neuropil synapses', top=25, log=True, normalize=False,
                     axis_fontsize=14, tick_fontsize=10):
    """Log (or linear) heatmap of a subtype x neuropil matrix. Mirrors 5b's styling."""
    from matplotlib.colors import LogNorm
    mat = mat.loc[mat.sum(axis=1).sort_values(ascending=False).index[:top]]
    plt.figure(figsize=(6, max(4, 0.32 * len(mat))))
    if log:
        data = mat.replace(0, np.nan)
        vmin = max(1, np.nanmin(data.values))
        sns.heatmap(data, cmap='magma', norm=LogNorm(vmin=vmin, vmax=np.nanmax(data.values)),
                    linewidths=.3, linecolor='lightgray')
        scale = 'log count'
    else:
        if normalize:
            mat = mat / mat.sum(axis=0)
        sns.heatmap(mat, cmap='magma')
        scale = 'fraction' if normalize else 'count'
    plt.gca().grid(False)
    plt.gca().tick_params(labelsize=tick_fontsize)
    plt.title(f'{title} ({scale})')
    plt.xlabel('subtype', fontsize=axis_fontsize); plt.ylabel('neuropil', fontsize=axis_fontsize)
    plt.tight_layout(); plt.show()
    return mat

out2_neuropil = neuropil_heatmap(neuropil_matrix_conn(down.out2_connections, min_syn=10),
                                 title='2nd-order output (out2) neuropils', top=25, log=True)
out2_neuropil

### 5c. Partner super-class composition

What kinds of neurons each subtype talks to (central brain, optic lobe, descending, ...).
Descending neurons are the brain's output to motor centres, so they're especially informative.

In [ ]:
def class_composition(groups, fafb, side='out'):
    sc = fafb.classes.set_index('root_id')['super_class']
    rows = []
    for name, ng in groups.items():
        flow = ng.downflow if side == 'out' else ng.upflow
        comp = flow['root_id'].map(sc).value_counts(normalize=True).rename(name)
        rows.append(comp)
    comp = pd.concat(rows, axis=1).fillna(0).T
    comp.plot(kind='bar', stacked=True, figsize=(7, 4), colormap='tab20')
    plt.ylabel('fraction of partners')
    plt.title(f'Partner super_class ({"downstream" if side=="out" else "upstream"})')
    plt.legend(title='super_class', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout(); plt.show()
    return comp

class_out = class_composition(groups0, fafb, side='out')
class_out

### 5d. Partner-overlap similarity (presynaptic vs postsynaptic, side by side)

Each subtype is a synapse-weighted vector over its partners; cosine similarity between
subtypes is then 0 = completely different partners, 1 = identical partners.

- `order=1` compares in1 (direct presynaptic) vs out1 (direct postsynaptic) partners.
- `order=2` compares in2 vs out2 (second-order partners).

Reads from the `up` and `down` Connectsets, so partner definitions match the in1/out1
heatmaps. Needs `down` built with `downstream_order=2` (section 3) and `up` with
`upstream_order=2` (section 4).

In [ ]:
def _cosine_sim(conn_dict, partner_col, min_syn):
    """Cosine similarity between subtypes from their synapse-weighted partner vectors.

    conn_dict:    {subtype: edge DataFrame}  (e.g. down.out1_connections)
    partner_col:  which id column identifies the partner ('post_root_id' or 'pre_root_id')."""
    vecs = {}
    for name, edges in conn_dict.items():
        vecs[name] = edges.query('syn_count >= @min_syn').groupby(partner_col)['syn_count'].sum()
    mat = pd.DataFrame(vecs).fillna(0)
    X = mat.values
    norm = np.linalg.norm(X, axis=0)
    norm[norm == 0] = 1
    sim = (X.T @ X) / np.outer(norm, norm)
    return pd.DataFrame(sim, index=mat.columns, columns=mat.columns)


def partner_similarity(up, down, order=1, min_syn=5, label_fontsize=20,
                       annot_fontsize=18, cbar_fontsize=14):
    """Plot input (presynaptic) and output (postsynaptic) partner similarity side by side.

    order=1 -> in1 / out1 ;  order=2 -> in2 / out2. Returns (input_sim, output_sim)."""
    if order == 1:
        in_conn, out_conn = up.in1_connections, down.out1_connections
    elif order == 2:
        in_conn, out_conn = up.in2_connections, down.out2_connections
    else:
        raise ValueError('order must be 1 or 2')
    in_sim  = _cosine_sim(in_conn,  'pre_root_id',  min_syn)
    out_sim = _cosine_sim(out_conn, 'post_root_id', min_syn)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    panels = [(f'input (in{order})', in_sim), (f'output (out{order})', out_sim)]
    for ax, (lbl, sim) in zip(axes, panels):
        sns.heatmap(sim, annot=True, fmt='.2f', cmap='rocket_r', vmin=0, vmax=1, square=True,
                    ax=ax, annot_kws={'fontsize': annot_fontsize})
        ax.set_title(f'Cosine similarity — {lbl}', pad=20)
        ax.tick_params(labelsize=label_fontsize)
        ax.collections[0].colorbar.ax.tick_params(labelsize=cbar_fontsize)
    plt.tight_layout(); plt.show()
    return in_sim, out_sim

In [ ]:
# first-order: in1 vs out1 (direct partners)
sim_in1, sim_out1 = partner_similarity(up, down, order=1)

# second-order: in2 vs out2
sim_in2, sim_out2 = partner_similarity(up, down, order=2)

### 5e. Input/output polarity

Ratio of input to output synapses per subtype — integrative (input-heavy) vs. broadcasting.

In [ ]:
def io_polarity(groups, fafb):
    npd = fafb.neuropils.set_index('root_id')
    rows = []
    for name, ng in groups.items():
        sub = npd.loc[npd.index.isin(ng.neurons['root_id'])]
        rows.append({'subtype': name,
                     'input_synapses': sub['input_synapses'].sum(),
                     'output_synapses': sub['output_synapses'].sum()})
    df = pd.DataFrame(rows).set_index('subtype')
    df['io_ratio'] = df['input_synapses'] / df['output_synapses']
    return df

io_polarity(groups0, fafb)

## 6. Statistical test: do the subtypes' INPUT partners overlap differently from their OUTPUT partners?

We test set overlap of partner neuron identities (not the synapse-weighted heatmap values).
Partner sets come from the order-1 groups already built: `in1` (from `up`) and `out1` (from
`down`). For each direction the universe (chance level) is the union of all four subtypes'
partners in that direction.

**This is a single connectome (n = 1).** There is no biological replication, so we do not use
tests that assume independent replicates. Instead:

- **Hypergeometric test** (per subtype-pair): is the observed overlap larger than expected if
  the two partner sets were random draws from the universe? Reports significance; the
  **Jaccard index** is the matched effect size. Same math as gene-set enrichment.
- **Fisher's exact test** (input vs output): are partners more often shared (by >=2 subtypes)
  among inputs than among outputs? Run whole-system and once per subtype.
- **Paired Wilcoxon** across the 6 subtype-pairs: are the pairwise input Jaccards
  systematically higher/lower than the matched output Jaccards? (n = 6 pairs, so directional
  but low-powered.)

p-values across the many pairs/subtypes are corrected with Benjamini-Hochberg (FDR) -> `q`.
Treat everything as inference within this connectome, and always report the raw counts +
Jaccard alongside the p-values.

### 6a. The statistics functions

`get_partner_sets` pulls the partner-ID sets out of a Connectsets object at a given level.
`overlap_stats` runs all the tests and plots the two Jaccard matrices side by side.

In [ ]:
from itertools import combinations
from collections import Counter
from scipy.stats import hypergeom, fisher_exact, wilcoxon


def benjamini_hochberg(pvals):
    """FDR correction. Returns q-values in the same order as the input list."""
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    order = np.argsort(p)
    adj = p[order] * n / np.arange(1, n + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.clip(adj, 0, 1)
    return out


def get_partner_sets(connectset, lvl):
    """{subtype_name: set of partner root_ids} at a level code (-1 = in1, +1 = out1, ...)."""
    groups = connectset._groups_at(lvl)
    return {name: set(ng.neurons['root_id']) for name, ng in groups.items()}


def _jaccard(a, b):
    """|intersection| / |union|. 0 = disjoint, 1 = identical."""
    union = len(a | b)
    return len(a & b) / union if union else 0.0


def overlap_stats(in_sets, out_sets, alpha=0.05, order=1, label_fontsize=18):
    """Compare input-partner overlap vs output-partner overlap.

    in_sets / out_sets: dicts {subtype: set(partner ids)} (e.g. from get_partner_sets)."""
    names = list(in_sets)
    pairs = list(combinations(names, 2))

    N_in = len(set().union(*in_sets.values()))
    N_out = len(set().union(*out_sets.values()))

    # per-pair Jaccard + hypergeometric enrichment
    recs = []
    for a, b in pairs:
        Ai, Bi = in_sets[a], in_sets[b]
        Ao, Bo = out_sets[a], out_sets[b]
        ki, ko = len(Ai & Bi), len(Ao & Bo)
        # hypergeom.sf(k-1, M, n, N) = P(overlap >= k): M=universe, n=|A|, N=|B|
        p_in = hypergeom.sf(ki - 1, N_in, len(Ai), len(Bi))
        p_out = hypergeom.sf(ko - 1, N_out, len(Ao), len(Bo))
        recs.append({'pair': f'{a}-{b}',
                     'J_in': _jaccard(Ai, Bi), 'J_out': _jaccard(Ao, Bo),
                     'shared_in': ki, 'shared_out': ko,
                     'p_hyp_in': p_in, 'p_hyp_out': p_out})
    pair_df = pd.DataFrame(recs)
    qvals = benjamini_hochberg(list(pair_df['p_hyp_in']) + list(pair_df['p_hyp_out']))
    pair_df['q_hyp_in'] = qvals[:len(pair_df)]
    pair_df['q_hyp_out'] = qvals[len(pair_df):]

    # whole-system: are inputs more shared than outputs?
    def shared_unique(sets):
        c = Counter()
        for s in sets.values():
            c.update(s)
        shared = sum(v >= 2 for v in c.values())
        unique = sum(v == 1 for v in c.values())
        return shared, unique

    sh_in, uq_in = shared_unique(in_sets)
    sh_out, uq_out = shared_unique(out_sets)
    table = [[sh_in, uq_in], [sh_out, uq_out]]
    OR, p_fisher = fisher_exact(table)
    try:
        w_stat, w_p = wilcoxon(pair_df['J_in'], pair_df['J_out'])
    except ValueError:
        w_stat, w_p = np.nan, np.nan

    print('=== WHOLE SYSTEM (all four subtypes) ===')
    print(f'input  : {sh_in} shared / {uq_in} unique partners  (universe {N_in})')
    print(f'output : {sh_out} shared / {uq_out} unique partners  (universe {N_out})')
    print(f"Fisher's exact (shared:unique, input vs output): OR={OR:.3f}, p={p_fisher:.3g}")
    print(f'Paired Wilcoxon on 6 pairwise Jaccards: stat={w_stat}, p={w_p:.3g}')
    print(f'mean Jaccard  input={pair_df.J_in.mean():.3f}  output={pair_df.J_out.mean():.3f}')

    # per subtype: shared with any other subtype, input vs output
    per = []
    for X in names:
        oin = set().union(*[in_sets[o] for o in names if o != X])
        oout = set().union(*[out_sets[o] for o in names if o != X])
        si, ui = len(in_sets[X] & oin), len(in_sets[X] - oin)
        so, uo = len(out_sets[X] & oout), len(out_sets[X] - oout)
        _, pX = fisher_exact([[si, ui], [so, uo]])
        per.append({'subtype': X,
                    'shared_frac_in': si / (si + ui), 'shared_frac_out': so / (so + uo),
                    'p_fisher': pX})
    per_df = pd.DataFrame(per)
    per_df['q_fisher'] = benjamini_hochberg(per_df['p_fisher'])

    def jmat(sets):
        m = pd.DataFrame(0.0, index=names, columns=names)
        for a in names:
            for b in names:
                m.loc[a, b] = 1.0 if a == b else _jaccard(sets[a], sets[b])
        return m
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, (lbl, sets) in zip(axes, [(f'input (in{order})', in_sets), (f'output (out{order})', out_sets)]):
        sns.heatmap(jmat(sets), annot=True, fmt='.2f', cmap='rocket_r',
                    vmin=0, vmax=1, square=True, ax=ax)
        ax.set_title(f'Jaccard overlap — {lbl}')
        ax.tick_params(labelsize=label_fontsize)
    plt.tight_layout(); plt.show()

    return {'pairwise': pair_df, 'per_subtype': per_df,
            'whole': {'fisher_OR': OR, 'fisher_p': p_fisher,
                      'wilcoxon_p': w_p, 'mean_J_in': pair_df.J_in.mean(),
                      'mean_J_out': pair_df.J_out.mean()}}

### 6b. Run it

`up` must be built with `upstream_order >= 1` and `down` with `downstream_order >= 1`
(sections 3-4).

In [ ]:
in1_sets = get_partner_sets(up, -1)
out1_sets = get_partner_sets(down, 1)

results = overlap_stats(in1_sets, out1_sets)

print('\n--- pairwise overlap (J = Jaccard effect size; q = FDR-adjusted) ---')
print(results['pairwise'].round(4).to_string(index=False))
print('\n--- per-subtype shared fraction (input vs output) ---')
print(results['per_subtype'].round(4).to_string(index=False))

### 6c. Same overlap test for second-order partners (in2 / out2)

Identical analysis as 6b, on the second-order partner sets (in2 = level -2, out2 = level +2).
Needs `up` built with `upstream_order=2` and `down` with `downstream_order=2`.

Second-order pools are much larger and tend to converge on shared hubs, so the baseline
overlap (and Jaccard) is usually higher than first order. The informative signal is the
contrast between input and output, and between subtypes — not the absolute value.

In [ ]:
in2_sets = get_partner_sets(up, -2)
out2_sets = get_partner_sets(down, 2)

results2 = overlap_stats(in2_sets, out2_sets, order=2)

print('\n--- pairwise overlap, 2nd order (J = Jaccard; q = FDR-adjusted) ---')
print(results2['pairwise'].round(4).to_string(index=False))
print('\n--- per-subtype shared fraction, 2nd order (input vs output) ---')
print(results2['per_subtype'].round(4).to_string(index=False))

In [ ]:
# resolve the cell-type column name (varies across classification CSV versions)
label_col = 'class' if 'class' in fafb.classes.columns else (
    'cell_type' if 'cell_type' in fafb.classes.columns else (
        'type' if 'type' in fafb.classes.columns else None
    )
)

if label_col is None:
    raise ValueError("No cell-type column found in fafb.classes. Check the classification CSV columns.")

subtype_frames = []
for name, ng in [('DM', DM), ('L1', L1), ('P1', P1), ('P2', P2)]:
    df = ng.neurons[['root_id']].copy()
    df['NPF cell type'] = name
    df = df.merge(fafb.classes[['root_id', label_col]], on='root_id', how='left')
    df = df.rename(columns={label_col: 'Cell Type'})
    subtype_frames.append(df[['NPF cell type', 'root_id', 'Cell Type']])

cell_type_table = pd.concat(subtype_frames, ignore_index=True).sort_values(
    ['NPF cell type', 'root_id']
).reset_index(drop=True)

cell_type_table.to_csv('cell_type_table.csv', index=False)

cell_type_table

## Appendix: self-test on fake data

Runs the classes on a small random graph (no real connectome needed) to confirm the code
works after editing. Prints `self-test OK` on success.

In [ ]:
def _self_test():
    import tempfile, os
    rng = np.random.default_rng(0); N = 120
    ids = 720575940000000000 + np.arange(N)
    rows = [(ids[a], ids[b], rng.choice(['SMP_R', 'LH_R', 'FB']),
             int(rng.integers(1, 30)), rng.choice(['ACH', 'GABA', 'GLUT']))
            for a, b in (rng.choice(N, 2, replace=False) for _ in range(1500))]
    conn = pd.DataFrame(rows, columns=['pre_root_id', 'post_root_id', 'neuropil', 'syn_count', 'nt_type'])
    nps = pd.DataFrame({'root_id': ids, 'input synapses': rng.integers(10, 500, N),
                        'output synapses': rng.integers(10, 500, N)})
    clf = pd.DataFrame({'root_id': ids, 'side': rng.choice(['left', 'right'], N),
                        'super_class': rng.choice(['central', 'optic', 'descending'], N),
                        'class': rng.choice(['A', 'B'], N)})
    neu = pd.DataFrame({'root_id': ids, 'nt_type': rng.choice(['ACH', 'GABA'], N),
                        'group': rng.choice(['SMP.LH', np.nan], N)})
    d = tempfile.mkdtemp()
    sv = lambda x, n: (x.to_csv(os.path.join(d, n), index=False) or os.path.join(d, n))
    f = FafbData(sv(conn, 'c'), sv(nps, 'p'), sv(clf, 'l'), sv(neu, 'n'))
    gs = [Neuralgroup(pd.DataFrame({'root_id': ids[s]}), f, nm, 5)
          for nm, s in {'DM': slice(0, 5), 'L1': slice(5, 10),
                        'P1': slice(10, 15), 'P2': slice(15, 22)}.items()]
    cs = Connectsets(gs, f, downstream_order=2, upstream_order=2, order2_min_syn=10)
    assert all((e['syn_count'] >= 10).all() for e in cs.out2_connections.values() if len(e))
    print('self-test OK')

_self_test()